# BERDL Gene Browser Mockup

This notebook generates a searchable HTML table displaying all genes in the ADP1 genome with annotations, pangenome classification, neighborhood conservation, annotation confidence, and fitness phenotypes.

## Process Gene Data and Compute Operons

In [1]:
%run util.py
# Load genome data
genome_data = util.load("ADP1Genome")

# Extract gene information and sort by start coordinate
genes = []
for feature in genome_data['data']['features']:
    if feature.get('type') == 'gene':
        gene_id = feature['id']
        location = feature.get('location', [[]])[0]
        
        # Get start coordinate and direction
        if len(location) >= 3:
            contig = location[0]
            start = location[1]
            direction = location[2]  # '+' or '-'
            length = location[3] if len(location) > 3 else 0
        else:
            start = 0
            direction = '+'
            length = 0
        
        # Get aliases (locus_tag, old_locus_tag, gene name)
        locus_tag = gene_id
        old_locus_tag = ""
        gene_name = ""
        protein_id = ""
        
        for alias in feature.get('aliases', []):
            if alias[0] == 'locus_tag':
                locus_tag = alias[1]
            elif alias[0] == 'old_locus_tag':
                old_locus_tag = alias[1]
            elif alias[0] == 'gene':
                gene_name = alias[1]
            elif alias[0] == 'protein_id':
                protein_id = alias[1]
        
        # Get primary function
        functions = feature.get('functions', [])
        primary_function = functions[0] if functions else ""
        
        genes.append({
            'gene_id': gene_id,
            'locus_tag': locus_tag,
            'old_locus_tag': old_locus_tag,
            'gene_name': gene_name,
            'protein_id': protein_id,
            'start': start,
            'length': length,
            'direction': direction,
            'primary_function': primary_function
        })

# Sort by start coordinate
genes.sort(key=lambda x: x['start'])

print(f"Processed {len(genes)} genes")
print(f"First gene: {genes[0]['gene_id']} at position {genes[0]['start']}")
print(f"Last gene: {genes[-1]['gene_id']} at position {genes[-1]['start']}")

/Users/chenry/Dropbox/Projects/KBUtilLib/src
modelseedpy 0.4.2


2025-12-04 09:45:41,483 - __main__.NotebookUtil - INFO - Loaded configuration from: /Users/chenry/.kbutillib/config.yaml
2025-12-04 09:45:41,483 - __main__.NotebookUtil - INFO - Loaded 0 tokens from /Users/chenry/.tokens
2025-12-04 09:45:41,485 - __main__.NotebookUtil - INFO - Loaded kbase tokens from /Users/chenry/.kbase/token


<class 'kbutillib.ms_fba_utils.MSFBAUtils'>
<class 'kbutillib.ai_curation_utils.AICurationUtils'>
<class 'kbutillib.notebook_utils.NotebookUtils'>
<class 'kbutillib.kb_plm_utils.KBPLMUtils'>
<class 'kbutillib.kb_model_utils.KBModelUtils'>
<class 'kbutillib.kb_annotation_utils.KBAnnotationUtils'>
loading biochemistry database from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


2025-12-04 09:45:46,496 - __main__.NotebookUtil - INFO - ModelSEED database loaded from /Users/chenry/Dropbox/Projects/ModelSEEDDatabase


<class 'modelseedpy.biochem.modelseed_biochem.ModelSEEDDatabase'>


2025-12-04 09:45:46,904 - __main__.NotebookUtil - WARNING - BLAST tools not found. Install NCBI BLAST+ to use BLAST functionality. On Ubuntu/Debian: sudo apt-get install ncbi-blast+, On MacOS: brew install blast
2025-12-04 09:45:46,905 - __main__.NotebookUtil - INFO - Notebook environment detected
2025-12-04 09:45:46,906 - __main__.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/


cobrakbase 0.4.0
Processed 3235 genes
First gene: ACIAD_RS00005 at position 201
Last gene: ACIAD_RS16655 at position 3598157


In [2]:
# Assign operon groups based on contiguous translation directions
# Genes in the same operon (same direction in a row) get the same color

# Define a palette of distinct colors for operons
operon_colors = [
    '#E74C3C',  # Red
    '#3498DB',  # Blue
    '#2ECC71',  # Green
    '#9B59B6',  # Purple
    '#F39C12',  # Orange
    '#1ABC9C',  # Teal
    '#E91E63',  # Pink
    '#00BCD4',  # Cyan
    '#8BC34A',  # Light Green
    '#FF5722',  # Deep Orange
    '#673AB7',  # Deep Purple
    '#009688',  # Teal Dark
]

operon_id = 0
current_direction = None

for i, gene in enumerate(genes):
    if gene['direction'] != current_direction:
        # New operon starts
        operon_id += 1
        current_direction = gene['direction']
    
    gene['operon_id'] = operon_id
    gene['operon_color'] = operon_colors[operon_id % len(operon_colors)]

print(f"Identified {operon_id} operon groups")

Identified 962 operon groups


## Enrich Gene Data with Annotations and Phenotypes

In [4]:
genome_data = util.load("ADP1Genome")
gene_annotations = util.load("gene_term_hash_named")
pangenome_data = util.load("pangenome_data")
neighborhood_data = util.load("neighborhood_conservation")
confidence_data = util.load("annotation_confidence")
fitness_data = util.load("fitness_phenotypes")

# Enrich each gene with additional data
for gene in genes:
    gene_id = gene['gene_id']
    
    # Get annotations from gene_term_hash_named
    anno = gene_annotations.get(gene_id, {})
    
    # Extract published model reactions
    reactions = []
    if 'reactions' in anno:
        if isinstance(anno['reactions'], list):
            reactions = [r.split(':')[0] if ':' in str(r) else str(r) for r in anno['reactions']]
        elif isinstance(anno['reactions'], dict):
            reactions = list(anno['reactions'].keys())
    gene['reactions'] = '; '.join(reactions) if reactions else ''
    
    # Extract other annotations (GLM4EC, DRAM, Snekmer)
    other_annos = []
    for key, value in anno.items():
        if 'GLM4EC' in key and isinstance(value, dict):
            for v in value.values():
                other_annos.append(f"GLM4EC: {v}")
        elif 'DRAM' in key and isinstance(value, dict):
            for v in value.values():
                other_annos.append(f"DRAM: {v}")
        elif 'Snekmer' in key and isinstance(value, dict):
            for v in value.values():
                other_annos.append(f"Snekmer: {str(v).strip()}")
    gene['other_annotations'] = ' | '.join(other_annos[:3]) if other_annos else ''  # Limit to 3
    
    # Get Uniprot ID
    gene['uniprot_id'] = anno.get('uniprot_id', '')
    
    # Get pangenome data (from placeholder)
    pan = pangenome_data.get('data', {}).get(gene_id, {})
    pan_class = pan.get('class', 'unknown')
    pan_conservation = pan.get('conservation', 0)
    gene['pangenome'] = f"{pan_class} ({pan_conservation:.1f}%)" if pan_class != 'unknown' else 'N/A'
    
    # Get neighborhood conservation (from placeholder)
    neigh = neighborhood_data.get('data', {}).get(gene_id, {})
    neigh_score = neigh.get('conservation_score', 0)
    neigh_conserved = neigh.get('conserved_neighbors', 0)
    neigh_total = neigh.get('total_neighbors_checked', 0)
    gene['neighborhood'] = f"{neigh_score:.2f} ({neigh_conserved}/{neigh_total})" if neigh_total > 0 else 'N/A'
    
    # Get annotation confidence (from placeholder)
    conf = confidence_data.get('data', {}).get(gene_id, {})
    conf_level = conf.get('confidence', 'unknown')
    conf_score = conf.get('score', 0)
    gene['confidence'] = f"{conf_level} ({conf_score:.2f})" if conf_level != 'unknown' else 'N/A'
    
    # Get fitness phenotype (from placeholder)
    fit = fitness_data.get('data', {}).get(gene_id, {})
    fitness = fit.get('overall_fitness', None)
    essential = fit.get('essential', False)
    summary = fit.get('phenotype_summary', '')
    if fitness is not None:
        if essential:
            gene['phenotype'] = f"ESSENTIAL ({fitness:.2f}) - {summary}"
        else:
            gene['phenotype'] = f"{fitness:.2f} - {summary}"
    else:
        gene['phenotype'] = 'N/A'

print("Gene data enriched with annotations and phenotypes")

Gene data enriched with annotations and phenotypes


## Generate Searchable HTML Table

In [5]:
def generate_html_table(genes):
    """Generate a searchable HTML table with gene data."""
    
    html = '''
<!DOCTYPE html>
<html>
<head>
    <title>ADP1 Gene Browser</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 20px;
            background-color: #f5f5f5;
        }
        h1 {
            color: #2c3e50;
        }
        .search-container {
            margin-bottom: 20px;
            padding: 15px;
            background-color: white;
            border-radius: 5px;
            box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        }
        #searchInput {
            width: 300px;
            padding: 10px;
            font-size: 14px;
            border: 1px solid #ddd;
            border-radius: 4px;
        }
        #searchInput:focus {
            outline: none;
            border-color: #3498db;
        }
        .stats {
            margin-left: 20px;
            color: #666;
        }
        table {
            border-collapse: collapse;
            width: 100%;
            background-color: white;
            box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        }
        th, td {
            border: 1px solid #ddd;
            padding: 8px;
            text-align: left;
            font-size: 12px;
        }
        th {
            background-color: #2c3e50;
            color: white;
            position: sticky;
            top: 0;
            cursor: pointer;
        }
        th:hover {
            background-color: #34495e;
        }
        tr:nth-child(even) {
            background-color: #f9f9f9;
        }
        tr:hover {
            background-color: #e8f4f8;
        }
        .arrow {
            font-size: 18px;
            font-weight: bold;
        }
        .essential {
            background-color: #ffebee;
            color: #c62828;
            font-weight: bold;
        }
        a {
            color: #3498db;
            text-decoration: none;
        }
        a:hover {
            text-decoration: underline;
        }
        .core { color: #27ae60; font-weight: bold; }
        .auxiliary { color: #f39c12; font-weight: bold; }
        .flexible { color: #e74c3c; font-weight: bold; }
        .confidence-high { color: #27ae60; }
        .confidence-medium { color: #f39c12; }
        .confidence-low { color: #e74c3c; }
        .truncate {
            max-width: 200px;
            white-space: nowrap;
            overflow: hidden;
            text-overflow: ellipsis;
        }
        .truncate:hover {
            white-space: normal;
            overflow: visible;
        }
    </style>
</head>
<body>
    <h1>ADP1 Gene Browser</h1>
    <div class="search-container">
        <input type="text" id="searchInput" placeholder="Search genes..." onkeyup="filterTable()">
        <span class="stats" id="stats">Showing <span id="visibleCount">''' + str(len(genes)) + '''</span> of ''' + str(len(genes)) + ''' genes</span>
    </div>
    <table id="geneTable">
        <thead>
            <tr>
                <th onclick="sortTable(0)">Locus ID</th>
                <th onclick="sortTable(1)">Dir</th>
                <th onclick="sortTable(2)">Primary Function</th>
                <th onclick="sortTable(3)">Reactions</th>
                <th onclick="sortTable(4)">Other Annotations</th>
                <th onclick="sortTable(5)">Uniprot</th>
                <th onclick="sortTable(6)">Pangenome</th>
                <th onclick="sortTable(7)">Neighborhood</th>
                <th onclick="sortTable(8)">Confidence</th>
                <th onclick="sortTable(9)">Phenotype</th>
            </tr>
        </thead>
        <tbody>
'''
    
    # Generate table rows
    for gene in genes:
        # Direction arrow
        arrow = '↓' if gene['direction'] == '+' else '↑'
        arrow_html = f'<span class="arrow" style="color: {gene["operon_color"]}">{arrow}</span>'
        
        # Locus ID with RefSeq link
        locus_display = gene['locus_tag']
        if gene['gene_name']:
            locus_display = f"{gene['gene_name']} ({gene['locus_tag']})"
        refseq_link = f'https://www.ncbi.nlm.nih.gov/gene/?term={gene["locus_tag"]}'
        locus_html = f'<a href="{refseq_link}" target="_blank">{locus_display}</a>'
        
        # Uniprot link
        uniprot_html = ''
        if gene['uniprot_id']:
            uniprot_link = f'https://www.uniprot.org/uniprotkb/{gene["uniprot_id"]}/entry'
            uniprot_html = f'<a href="{uniprot_link}" target="_blank">{gene["uniprot_id"]}</a>'
        
        # Pangenome class styling
        pangenome_html = gene['pangenome']
        if 'core' in pangenome_html.lower():
            pangenome_html = f'<span class="core">{pangenome_html}</span>'
        elif 'auxiliary' in pangenome_html.lower():
            pangenome_html = f'<span class="auxiliary">{pangenome_html}</span>'
        elif 'flexible' in pangenome_html.lower():
            pangenome_html = f'<span class="flexible">{pangenome_html}</span>'
        
        # Confidence styling
        confidence_html = gene['confidence']
        if 'high' in confidence_html.lower():
            confidence_html = f'<span class="confidence-high">{confidence_html}</span>'
        elif 'medium' in confidence_html.lower():
            confidence_html = f'<span class="confidence-medium">{confidence_html}</span>'
        elif 'low' in confidence_html.lower():
            confidence_html = f'<span class="confidence-low">{confidence_html}</span>'
        
        # Phenotype styling
        phenotype_html = gene['phenotype']
        row_class = ''
        if 'ESSENTIAL' in phenotype_html:
            row_class = ' class="essential"'
        
        html += f'''            <tr{row_class}>
                <td>{locus_html}</td>
                <td style="text-align: center;">{arrow_html}</td>
                <td class="truncate" title="{gene['primary_function']}">{gene['primary_function']}</td>
                <td class="truncate" title="{gene['reactions']}">{gene['reactions']}</td>
                <td class="truncate" title="{gene['other_annotations']}">{gene['other_annotations']}</td>
                <td>{uniprot_html}</td>
                <td>{pangenome_html}</td>
                <td>{gene['neighborhood']}</td>
                <td>{confidence_html}</td>
                <td class="truncate" title="{phenotype_html}">{phenotype_html}</td>
            </tr>
'''
    
    html += '''        </tbody>
    </table>
    
    <script>
    function filterTable() {
        var input = document.getElementById("searchInput");
        var filter = input.value.toLowerCase();
        var table = document.getElementById("geneTable");
        var tr = table.getElementsByTagName("tr");
        var visibleCount = 0;
        
        for (var i = 1; i < tr.length; i++) {
            var td = tr[i].getElementsByTagName("td");
            var found = false;
            for (var j = 0; j < td.length; j++) {
                if (td[j]) {
                    var txtValue = td[j].textContent || td[j].innerText;
                    if (txtValue.toLowerCase().indexOf(filter) > -1) {
                        found = true;
                        break;
                    }
                }
            }
            if (found) {
                tr[i].style.display = "";
                visibleCount++;
            } else {
                tr[i].style.display = "none";
            }
        }
        document.getElementById("visibleCount").innerText = visibleCount;
    }
    
    var sortDirection = {};
    function sortTable(n) {
        var table = document.getElementById("geneTable");
        var rows = Array.from(table.rows).slice(1);
        var dir = sortDirection[n] === "asc" ? "desc" : "asc";
        sortDirection[n] = dir;
        
        rows.sort(function(a, b) {
            var x = a.cells[n].textContent.toLowerCase();
            var y = b.cells[n].textContent.toLowerCase();
            
            // Try numeric sort first
            var numX = parseFloat(x);
            var numY = parseFloat(y);
            if (!isNaN(numX) && !isNaN(numY)) {
                return dir === "asc" ? numX - numY : numY - numX;
            }
            
            // Fall back to string sort
            if (dir === "asc") {
                return x.localeCompare(y);
            } else {
                return y.localeCompare(x);
            }
        });
        
        var tbody = table.getElementsByTagName("tbody")[0];
        rows.forEach(function(row) {
            tbody.appendChild(row);
        });
    }
    </script>
</body>
</html>
'''
    
    return html

# Generate the HTML
html_content = generate_html_table(genes)

# Save to file
output_path = "nboutput/gene_browser.html"
with open(output_path, 'w') as f:
    f.write(html_content)

print(f"HTML table saved to {output_path}")
print(f"Open this file in a web browser to view the searchable gene table.")

HTML table saved to nboutput/gene_browser.html
Open this file in a web browser to view the searchable gene table.


## Preview Table (First 50 Genes)

In [6]:
# Display a preview of the first 50 genes in the notebook
preview_html = generate_html_table(genes[:50])
display(HTML(preview_html))

Locus ID,Dir,Primary Function,Reactions,Other Annotations,Uniprot,Pangenome,Neighborhood,Confidence,Phenotype
dnaA (ACIAD_RS00005),↓,Chromosomal replication initiator protein DnaA,rxn05294_c0,DRAM: chromosomal replication initiator protein | Snekmer: DnaA N-terminal domain,B2HZA7,core (98.5%),0.95 (8/10),high (0.98),ESSENTIAL (0.12) - Essential - severe growth defect in all conditions
ACIAD_RS00010,↓,Beta sliding clamp,rxn05294_c0,"GLM4EC: DNA-directed DNA polymerase. | DRAM: DNA polymerase III subunit beta [EC:2.7.7.7] | Snekmer: DNA polymerase III beta subunit, C-terminal domain",P43744,core (99.2%),0.92 (7/10),high (0.95),0.95 - Near-wildtype growth
recF (ACIAD_RS00015),↓,DNA replication and repair protein RecF,,"DRAM: DNA replication and repair protein RecF | Snekmer: AAA domain, putative AbiEii toxin, Type IV TA system",A3M0Q6,core (97.8%),0.88 (6/10),high (0.92),0.88 - Mild growth defect
gyrB (ACIAD_RS00020),↓,DNA gyrase subunit B,rxn05294_c0,GLM4EC: 1 | DRAM: DNA gyrase subunit B [EC:5.99.1.3] | Snekmer: Toprim domain,P0AES7,core (99.5%),0.97 (9/10),high (0.97),ESSENTIAL (0.05) - Essential - lethal in most conditions
ACIAD_RS00025,↑,DUF2171 domain-containing protein,,Snekmer: Uncharacterized protein conserved in bacteria (DUF2171),Q73BA6,auxiliary (72.3%),0.65 (4/10),low (0.35),1.02 - No phenotype detected
ACIAD_RS00030,↑,putative ABC transporter ATP-binding protein YheS,TRANS_DASH_RXNIM_DASH_5895(rxn05145_c0),"GLM4EC: 1 | DRAM: ATP-binding cassette, subfamily F, member 3 | Snekmer: ABC transporter",P63389,core (95.1%),0.78 (5/10),medium (0.72),0.72 - Moderate growth defect
ACIAD_RS00035,↓,AdeT,,Snekmer: Acetyltransferase (GNAT) domain,Q122C7,flexible (45.2%),0.42 (2/10),low (0.28),0.98 - No significant phenotype
ACIAD_RS00040,↓,AdeT,,Snekmer: Phosphotransferase enzyme family,P16700,flexible (38.7%),0.35 (2/10),low (0.31),1.00 - No phenotype detected
erpA (ACIAD_RS00045),↓,Iron-sulfur cluster insertion protein ErpA,,DRAM: iron-sulfur cluster insertion protein | Snekmer: Iron-sulphur cluster biosynthesis,B0VML0,core (96.8%),0.85 (6/10),high (0.89),0.45 - Severe growth defect
ACIAD_RS00050,↑,Anhydro-N-acetylmuramic acid kinase,rxn08136_c0,GLM4EC: Anhydro-N-acetylmuramic acid kinase. | DRAM: anhydro-N-acetylmuramic acid kinase [EC:2.7.1.170] | Snekmer: Anhydro-N-acetylmuramic acid kinase,Q48NN5,auxiliary (68.4%),0.72 (5/10),medium (0.68),0.82 - Mild to moderate growth defect
